In [1]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from dataset import train_loader, class_names


# 1. Вибір пристрою
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Пристрій:", device)


# 2. Завантаження попередньо навченої ResNet18
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)


# 3. Заморожування параметрів базової мережі
for param in model.parameters():
    param.requires_grad = False


# 4. Заміна класифікаційного шару
model.fc = nn.Linear(
    model.fc.in_features,
    len(class_names)
)


# 5. Переміщення моделі на пристрій
model = model.to(device)


# 6. Функція втрат
criterion = nn.CrossEntropyLoss()


# 7. Оптимізатор
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.0001
)


# 8. Навчання
epochs = 5

for epoch in range(epochs):

    model.eval()
    model.fc.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Обнулення градієнтів
        optimizer.zero_grad()

        # Пряме поширення
        outputs = model(images)

        # Обчислення помилки
        loss = criterion(outputs, labels)

        # Зворотне поширення
        loss.backward()

        # Оновлення ваг
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        # Передбачення класу
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


# 9. Збереження моделі
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_to_idx": train_loader.dataset.class_to_idx
    },
    "pneumonia_resnet18.pth"
)

print("\nМодель успішно збережена!")


Класи: ['NORMAL', 'PNEUMONIA']
Відповідність класів: {'NORMAL': 0, 'PNEUMONIA': 1}
Train: 5216
Test: 624
Validation: 16
Пристрій: cpu
Epoch [1/5] Loss: 0.4899 Accuracy: 76.61%
Epoch [2/5] Loss: 0.2834 Accuracy: 91.07%
Epoch [3/5] Loss: 0.2154 Accuracy: 93.37%
Epoch [4/5] Loss: 0.1843 Accuracy: 93.94%
Epoch [5/5] Loss: 0.1653 Accuracy: 94.36%

Модель успішно збережена!
